In [3]:
query = """
WITH SalesCTE AS (
    -- Собираем нужные данные через JOIN
    SELECT
        m.manager,
        c.region,
        SUM(f.total_revenue_rub) as total_sales
    FROM fact_sales f
    LEFT JOIN dim_managers m ON f.manager_id = m.manager_id
    LEFT JOIN dim_clients c ON f.client_id = c.client_id
    WHERE f.status = 'Успешно реализовано'
    GROUP BY m.manager, c.region
)
SELECT
    manager,
    region,
    total_sales,
    -- Ранжируем менеджеров внутри каждого региона
    RANK() OVER(PARTITION BY region ORDER BY total_sales DESC) as regional_rank,
    -- Считаем долю продаж менеджера от общих продаж в регионе
    ROUND(total_sales / SUM(total_sales) OVER(PARTITION BY region) * 100, 2) as region_share_percent
FROM SalesCTE
ORDER BY region, regional_rank;
"""

analytics_df = pd.read_sql_query(query, conn)
display(analytics_df.head(10))

,manager,region,total_sales,regional_rank,region_share_percent
0,Попова,Екатеринбург,1.123055e+08,1,27.18
1,Иванов,Екатеринбург,1.046103e+08,2,25.32
2,Смирнов,Екатеринбург,9.947264e+07,3,24.07
3,Кузнецов,Екатеринбург,9.680319e+07,4,23.43
4,Смирнов,Казань,1.193624e+08,1,29.38
5,Иванов,Казань,1.041531e+08,2,25.64
6,Попова,Казань,9.506932e+07,3,23.40
7,Кузнецов,Казань,8.768535e+07,4,21.58
8,Кузнецов,Москва,1.053139e+08,1,28.61
9,Попова,Москва,9.663934e+07,2,26.25
